# Master Ensemble v1

Combines every approach from `Experimentation/Final_Ensemble/` -- 5 LLM fine-tunes, the Embedding ensemble, and the Cross-Encoder ensemble -- into one notebook: a shared job allocator lets you pick which models to include, how many seeds/members each, and the relative **weight** each approach gets in the final blend. It then either **AUTOMATIC**-schedules jobs across Kaggle's 2x T4 for minimum wall time, or lets you assign GPUs **MANUALLY**, previews the resulting timeline against the 12h competition limit, executes it for real, and blends the results into `submission.csv`.

In [ ]:
import os
import sys
IS_KAGGLE_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

skip_presub_run = 1

if not IS_KAGGLE_SUBMISSION and skip_presub_run:
    os.system("cp /kaggle/input/jigsaw-agile-community-rules/sample_submission.csv /kaggle/working/submission.csv")
    sys.exit()

In [ ]:
%%writefile constants.py
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"
LORA_BASE_DIR = "output/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = f"Reddit moderation: Does the comment violate the rule? Answer '{POSITIVE_ANSWER}' or '{NEGATIVE_ANSWER}' only."

# ==============================================================================
# Model registry -- one entry per approach from Final_Ensemble/README.md.
#
# tp_mode:
#   "shared"           -> each seed trains + infers on ONE pinned GPU
#                         (tensor_parallel_size=1); multiple seeds can run
#                         concurrently on separate GPUs.
#   "tensor_parallel"   -> each seed trains on ONE pinned GPU, then infers
#                         with the model split across every visible GPU
#                         (tensor_parallel_size=num_gpus) -- needs BOTH GPUs
#                         exclusively for the duration of inference.
#
# train_min / infer_min are the per-seed wall-clock estimates from
# README.md (a manually-provided combined figure, split in half) -- used
# only for the schedule preview. Real runs will differ; the executor uses
# real completion times regardless of these estimates.
# ==============================================================================
MODEL_REGISTRY = {
    "lama3-8b": {
        "name": "lama3-8b-INSTRU",
        "base_model_path": "/kaggle/input/models/shivamreturns/unsloth-meta-llama-3-1-8b-instruct/transformers/default/1/Meta-Llama-3.1-8B-Instruct",
        "chat_template": "llama-3.1",
        "tp_mode": "tensor_parallel",
        "gpu_memory_utilization": 0.98,
        "pbl": 0.926, "pvt": 0.921,
        "train_min": 48, "infer_min": 25,
    },
    "qwen3-8b": {
        "name": "QWEN3-8b",
        "base_model_path": "/kaggle/input/models/shivamreturns/unsloth-qwen3-8b/transformers/default/1/Qwen3-8B",
        "chat_template": "qwen-3",
        "tp_mode": "tensor_parallel",
        "gpu_memory_utilization": 0.98,
        "pbl": 0.924, "pvt": 0.921,
        "train_min": 48, "infer_min": 25,
    },
    "qwen3-4b": {
        "name": "QWEN3-4b-INSTRU",
        "base_model_path": "/kaggle/input/models/wowfattie/qwen3-4b-instruct-2507/transformers/default/1",
        "chat_template": "qwen3-instruct",
        "tp_mode": "shared",
        "gpu_memory_utilization": 0.9,
        "pbl": 0.924, "pvt": 0.918,
        "train_min": 26, "infer_min": 24,
    },
    "qwen25-7b": {
        "name": "QWEN2.5-7b-INSTRU",
        "base_model_path": "/kaggle/input/models/shivamreturns/unsloth-qwen2-5-7b/transformers/default/1/Qwen2.5-7B",
        "chat_template": "qwen-2.5",
        "tp_mode": "tensor_parallel",
        "gpu_memory_utilization": 0.98,
        "pbl": 0.923, "pvt": 0.919,
        "train_min": 48, "infer_min": 25,
    },
    "llama32-3b": {
        "name": "llama3.2-3b-INSTRU",
        "base_model_path": "/kaggle/input/models/shivamreturns/unsloth-llama-3-2-3b-instruct/transformers/default/1/Llama-3.2-3B-Instruct",
        "chat_template": "llama-3.2",
        "tp_mode": "shared",
        "gpu_memory_utilization": 0.9,
        "pbl": 0.923, "pvt": 0.917,
        "train_min": 26, "infer_min": 24,
    },
}

# Embedding is a different (non-LLM) pipeline: ONE script does both training
# AND inference for each of 3 ensemble members (m1/m2/m3) -- modeled as a
# single combined run per member (see train_embedding_single.py), not
# separate train/infer stages.
EMBEDDING_MEMBERS = [
    {"label": "m1", "model_path": "/kaggle/input/notebooks/shivamreturns/embeddingmodels-repo/downloaded_models/thenlper__gte-large/", "save_dir": "reddit-semantic-model_m1", "output_csv": "submission_m1.csv"},
    {"label": "m2", "model_path": "/kaggle/input/models/jonathanchan/baai/transformers/bge-large-en-v1.5/1", "save_dir": "reddit-semantic-model_m2", "output_csv": "submission_m2.csv"},
    {"label": "m3", "model_path": "/kaggle/input/notebooks/shivamreturns/embeddingmodels-repo/downloaded_models/intfloat__e5-large-v2/", "save_dir": "reddit-semantic-model_m3", "output_csv": "submission_m3.csv"},
]
EMBEDDING_INFO = {"name": "Embedding", "pbl": 0.916, "pvt": 0.910, "run_min": 15}

# Cross-Encoder is a THIRD distinct pipeline: 3 different base encoder
# models (not 3 seeds of one model), each trained via sentence-transformers'
# CrossEncoderTrainer. Like Embedding, one process does both training AND
# inference per member (see train_cross_encoder_single.py) -- modeled as a
# single combined run per member, not separate train/infer stages.
CROSS_ENCODER_MEMBERS = [
    {"label": "m1", "model_path": "/kaggle/input/cross-encoders-v2/downloaded_models/google__electra-base-discriminator/", "seed": 42},
    {"label": "m2", "model_path": "/kaggle/input/cross-encoders-v1/downloaded_models/FacebookAI__roberta-base/", "seed": 123},
    {"label": "m3", "model_path": "/kaggle/input/huggingfacedebertav3variants/deberta-v3-base", "seed": 456},
]
CROSS_ENCODER_INFO = {"name": "Cross-Encoder", "pbl": 0.912, "pvt": 0.904, "run_min": 10}
CROSS_ENCODER_BATCH_SIZE = 32
CROSS_ENCODER_EPOCHS = 3
CROSS_ENCODER_LEARNING_RATE = 2e-5
CROSS_ENCODER_MAX_LENGTH = 512


def lora_path_for(model_id, seed):
    return f"{LORA_BASE_DIR}{model_id}/seed_{seed}/"


def pred_path_for(model_id, seed):
    return f"preds_{model_id}_seed_{seed}.csv"


def ce_save_dir_for(label):
    return f"reddit-cross-encoder-model-{label}"


def ce_pred_path_for(label):
    return f"preds_ce_{label}.csv"

In [ ]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT


def build_prompt(row):
    """Variation 4: Numbered format"""
    return f"""
Reddit moderation: Does the comment violate the rule? Answer '{POSITIVE_ANSWER}' or '{NEGATIVE_ANSWER}' only.

1. Rule: {row["rule"]}
2. Comment: {row["body"]}
---
Answer:"""


def get_dataframe_to_train(data_path: str) -> pd.DataFrame:
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = []

    # base train rows
    base = train_dataset[["body", "rule", "rule_violation"]].copy()
    base["source"] = "train"
    flatten.append(base)

    # Upsample target block (test examples) later by labeling them now
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            col = f"{violation_type}_example_{i}"
            sub_dataset = test_dataset[[col, "rule"]].copy()
            sub_dataset = sub_dataset.rename(columns={col: "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            sub_dataset["source"] = "test_examples"
            flatten.append(sub_dataset)

    # combine & dedupe first (so oversampling isn't undone)
    dataframe = pd.concat(flatten, axis=0, ignore_index=True)
    dataframe = dataframe.drop_duplicates(ignore_index=True)

    # upsample test_examples to appear 3x total (add two extra copies)
    test_rows = dataframe[dataframe["source"] == "test_examples"]
    if not test_rows.empty:
        dataframe = pd.concat([dataframe, test_rows], axis=0, ignore_index=True)

    # optional: shuffle for randomness
    dataframe = dataframe.sample(frac=1.0, random_state=1001).reset_index(drop=True)

    # drop helper column before returning
    dataframe = dataframe.drop(columns=["source"])
    return dataframe


def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    return dataset

In [ ]:
%%writefile train_llm.py

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # only used if the launcher didn't already pin a GPU; must come before any torch/unsloth import

import warnings
warnings.filterwarnings("ignore")
import transformers
transformers.logging.set_verbosity_error()

import time
import random
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

MODEL_ID = os.environ["MODEL_ID"]
SEED = int(os.environ.get("SEED", 1010))

print("PID:", os.getpid(), "CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"), "MODEL_ID:", MODEL_ID, "SEED:", SEED)
import torch
import numpy as np
assert torch.cuda.is_available()
print("Visible device count:", torch.cuda.device_count())
torch.cuda.set_device(0)            # 0 == the only GPU visible to this process
print("Using:", torch.cuda.get_device_name(0))


def set_seed(seed_value=SEED):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)

set_seed()

os.environ.setdefault("HF_HUB_OFFLINE", "1")          # never hit the Hub
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")     # Transformers offline
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")      # Datasets offline
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1") # no telemetry

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, MODEL_REGISTRY, lora_path_for


def main():
    t_start = time.perf_counter()
    cfg = MODEL_REGISTRY[MODEL_ID]
    lora_path = lora_path_for(MODEL_ID, SEED)
    os.makedirs(lora_path, exist_ok=True)

    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)

    print(cfg["base_model_path"])

    quant_mode = os.environ.get("QUANT_MODE", "4bit").lower()   # "4bit" or "8bit"
    load_in_4bit = (quant_mode == "4bit")
    dtype = None  # auto (fp16 on T4/V100, bf16 on Ampere+)

    t0 = time.perf_counter()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = cfg["base_model_path"],
        max_seq_length = 512,
        dtype          = dtype,
        load_in_4bit   = load_in_4bit,
        load_in_8bit   = (quant_mode == "8bit"),
        local_files_only=True,
    )

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )

    tokenizer = get_chat_template(tokenizer, chat_template=cfg["chat_template"])
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.truncation_side = "left"
    t_load = time.perf_counter() - t0

    t0 = time.perf_counter()
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        max_seq_length=256,
        packing=False,
        args=SFTConfig(
            per_device_train_batch_size=16,
            gradient_accumulation_steps=1,
            num_train_epochs=1,
            learning_rate=1.5e-4,
            weight_decay=0.01,
            lr_scheduler_type="linear",
            warmup_steps=0,
            logging_steps=10,
            optim="adamw_8bit",
            seed=SEED,
            save_strategy="no",
            report_to="none",
            dataloader_num_workers=2,
            completion_only_loss=True,
        ),
    )
    t_setup = time.perf_counter() - t0

    t0 = time.perf_counter()
    trainer.train()
    t_train = time.perf_counter() - t0

    t0 = time.perf_counter()
    trainer.save_model(lora_path)
    t_save = time.perf_counter() - t0

    t_total = time.perf_counter() - t_start
    print("\n" + "=" * 50)
    print(f"TRAINING TIME BENCHMARK (model={MODEL_ID}, seed={SEED})")
    print("=" * 50)
    print(f"lora output path     : {lora_path}")
    print(f"training examples    : {len(train_dataset)}")
    print(f"model+LoRA load time : {t_load:.2f}s")
    print(f"trainer setup time   : {t_setup:.2f}s")
    print(f"trainer.train() time : {t_train:.2f}s")
    print(f"save_model() time    : {t_save:.2f}s")
    print(f"total wall time      : {t_total:.2f}s")
    print("=" * 50)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile infer_llm.py
import os
os.environ["VLLM_USE_V1"] = "0"

MODEL_ID = os.environ["MODEL_ID"]
SEED = int(os.environ.get("SEED", 1010))

import time
import math
import vllm
import torch
import numpy as np
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER, MODEL_REGISTRY, lora_path_for, pred_path_for

POSITIVE_VARIANTS = ["Yes", "YES", "Y", "yes", "True"]
NEGATIVE_VARIANTS = ["No", "NO", "N", "no", "False"]


def build_variant_choices(variants):
    """Bare + space-prefixed spelling for each variant, matching Rank1's
    _first_token_ids helper -- covers cases where the tokenizer represents
    e.g. "Yes" and " Yes" as different tokens.
    """
    choices = []
    for v in variants:
        choices.append(v)
        choices.append(" " + v)
    return choices


def logsumexp(values):
    if not values:
        return float("-inf")
    m = max(values)
    return m + math.log(sum(math.exp(v - m) for v in values))


def main():
    cfg = MODEL_REGISTRY[MODEL_ID]
    num_gpus = max(torch.cuda.device_count(), 1)
    tp_size = num_gpus if cfg["tp_mode"] == "tensor_parallel" else 1
    print("PID:", os.getpid(), "CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"),
          "MODEL_ID:", MODEL_ID, "SEED:", SEED, "GPUs visible:", num_gpus, "tensor_parallel_size:", tp_size)
    t_start = time.perf_counter()

    t0 = time.perf_counter()
    llm = vllm.LLM(
        cfg["base_model_path"],
        tensor_parallel_size=tp_size,
        gpu_memory_utilization=cfg["gpu_memory_utilization"],
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=512,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )
    t_load = time.perf_counter() - t0

    tokenizer = llm.get_tokenizer()

    positive_choices = build_variant_choices(POSITIVE_VARIANTS)
    negative_choices = build_variant_choices(NEGATIVE_VARIANTS)
    all_choices = positive_choices + negative_choices

    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=all_choices)
    positive_ids = set(mclp.choice_tokens[:len(positive_choices)])
    negative_ids = set(mclp.choice_tokens[len(positive_choices):])
    n_candidates = len(set(mclp.choice_tokens))
    print(f"Positive token ids: {sorted(positive_ids)}")
    print(f"Negative token ids: {sorted(negative_ids)}")

    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataset = build_dataset(test_dataframe)
    texts = test_dataset["prompt"]

    lora_path = lora_path_for(MODEL_ID, SEED)

    t0 = time.perf_counter()
    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=min(n_candidates, 20),  # vLLM hard-caps logprobs at 20
        ),
        use_tqdm=True,
        lora_request=LoRARequest(f"{MODEL_ID}_seed_{SEED}", SEED, lora_path),
    )
    t_generate = time.perf_counter() - t0

    p_yes_list = []
    for out in outputs:
        logprob_by_id = out.outputs[0].logprobs[0]  # dict: token_id -> Logprob
        yes_lp = [lp.logprob for tid, lp in logprob_by_id.items() if tid in positive_ids]
        no_lp = [lp.logprob for tid, lp in logprob_by_id.items() if tid in negative_ids]
        y = logsumexp(yes_lp)
        n = logsumexp(no_lp)
        m = max(y, n)
        p_yes_list.append(math.exp(y - m) / (math.exp(y - m) + math.exp(n - m)))

    # Per-rule ranked score in [0, 1] for THIS model+seed (matches Rank1's
    # approach) -- computed per seed, before combining, so the ensemble
    # averages calibrated per-rule ranks rather than raw probabilities that
    # may be calibrated differently model to model.
    df_scores = pd.DataFrame({
        "row_id": test_dataframe["row_id"],
        "rule": test_dataframe["rule"],
        "prob": p_yes_list,
    })
    grp = df_scores.groupby("rule")
    rank = grp["prob"].rank(method="average", ascending=True)
    n_group = grp["prob"].transform("size")
    df_scores["rule_violation"] = (rank - 1.0) / np.maximum(n_group - 1.0, 1.0)

    pred_path = pred_path_for(MODEL_ID, SEED)
    df_scores[["row_id", "rule_violation"]].to_csv(pred_path, index=False)

    t_total = time.perf_counter() - t_start
    n_rows = len(test_dataframe)
    print("\n" + "=" * 50)
    print(f"INFERENCE TIMING (model={MODEL_ID}, seed={SEED})")
    print("=" * 50)
    print(f"pred output path : {pred_path}")
    print(f"rows processed   : {n_rows}")
    print(f"model load time  : {t_load:.2f}s")
    print(f"generate() time  : {t_generate:.2f}s")
    print(f"total wall time  : {t_total:.2f}s")
    print("=" * 50)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_embedding_single.py
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# setdefault() only kicks in when unset, so the orchestrator's per-job CUDA_VISIBLE_DEVICES
# assignment is never overridden -- this is just a safety net for standalone testing of
# this single script (an unset CUDA_VISIBLE_DEVICES would let sentence-transformers' Trainer
# auto-detect both GPUs and silently double the effective batch size).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print("PID:", os.getpid(), "CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
import torch
assert torch.cuda.is_available()
print("Visible device count:", torch.cuda.device_count())
torch.cuda.set_device(0)            # 0 == the ONLY visible GPU in this process
print("Using:", torch.cuda.get_device_name(0))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ["WANDB_DISABLED"] = "true"

import sys
import gc
import random
import time
import inspect
import subprocess
import threading
import numpy as np
import pandas as pd
import sentence_transformers
from cleantext import clean
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, losses, util, InputExample
from sentence_transformers.util import semantic_search, dot_score
from torch.utils.data import DataLoader

# =========================
# Constants
# =========================
DATA_PATH = os.environ.get("DATA_PATH", "/kaggle/input/jigsaw-agile-community-rules")
CLEAN_TEXT = True
TOP_K = 50
BATCH_SIZE = 128          # encoding batch size (inference)
TRAIN_BATCH_SIZE = 16     # triplet fine-tuning batch size

STEP_TIMINGS = []  # list of {"stage": ..., "model": ..., "seconds": ...}

def record_step(stage, model, seconds):
    STEP_TIMINGS.append({"stage": stage, "model": model, "seconds": round(seconds, 3)})

class GpuUtilSampler:
    """Samples `nvidia-smi` GPU utilization % in a background thread for the duration of a `with` block."""
    def __init__(self, interval=1.0):
        self.interval = interval
        self._stop = threading.Event()
        self._samples = []
        self._thread = None

    def _poll(self):
        while not self._stop.is_set():
            try:
                out = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                    timeout=2,
                ).decode().strip()
                self._samples.append(int(out.splitlines()[0]))
            except Exception:
                pass
            self._stop.wait(self.interval)

    def __enter__(self):
        self._thread = threading.Thread(target=self._poll, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        self._thread.join(timeout=2)

    def summary(self):
        if not self._samples:
            return "no samples captured"
        return (f"avg={sum(self._samples)/len(self._samples):.0f}% "
                f"min={min(self._samples)}% max={max(self._samples)}% (n={len(self._samples)} samples)")

def print_environment_diagnostics():
    print("=== Environment diagnostics ===")
    print("torch version:", torch.__version__)
    print("sentence_transformers version:", sentence_transformers.__version__)
    fit_params = inspect.signature(SentenceTransformer.fit).parameters
    amp_is_real_param = "use_amp" in fit_params
    print(f"SentenceTransformer.fit() accepts 'use_amp' as an explicit parameter: {amp_is_real_param}")
    if not amp_is_real_param:
        print("  -> WARNING: 'use_amp' is NOT a recognized parameter in this installed version's fit()."
              " It is likely being silently swallowed and doing nothing.")

# =========================
# Data utils
# =========================
def build_prompt(row):
    return f"""{row["body"]}"""

def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )

def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")

    flatten = [train_dataset[["body", "rule", "subreddit", "rule_violation"]]]
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=["body", "rule"], ignore_index=True)
    return dataframe

def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)
    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map({1: 1, 0: -1})
    return dataframe

# =========================
# Fine-tuning (triplet loss on rule / violating / non-violating comment)
# =========================
def train_embedding_model(model_path, save_path, corpus_dataframe):
    t0 = time.perf_counter()
    data = corpus_dataframe
    data_by_rule_and_status = data.groupby(["rule", "rule_violation"])["body"].apply(list).to_dict()
    record_step("group_by_rule_status", save_path, time.perf_counter() - t0)

    t0 = time.perf_counter()
    triplet_examples = []
    for violating_row in data[data["rule_violation"] == True].itertuples():
        anchor_rule = violating_row.rule
        positive_comment = violating_row.body
        negatives = data_by_rule_and_status.get((anchor_rule, False), [])
        if negatives:
            sampled_negatives = random.sample(negatives, k=min(2, len(negatives)))
            while len(sampled_negatives) < 2:
                sampled_negatives.append(random.choice(negatives))
            for negative_comment in sampled_negatives:
                triplet_examples.append(InputExample(texts=[anchor_rule, positive_comment, negative_comment]))
        else:
            print(f"Skipping rule '{anchor_rule}' as no non-violating comments were found for it.")
    record_step("build_triplets", save_path, time.perf_counter() - t0)

    print(f"Generated {len(triplet_examples)} triplet examples.")

    t0 = time.perf_counter()
    model = SentenceTransformer(model_name_or_path=model_path, device="cuda")
    record_step("load_base_model", save_path, time.perf_counter() - t0)
    print("Model loaded successfully.")

    train_dataloader = DataLoader(triplet_examples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
    print(f"DataLoader created with batch size: {train_dataloader.batch_size}")

    train_loss = losses.TripletLoss(model=model, triplet_margin=0.3, distance_metric=losses.TripletDistanceMetric.COSINE)
    print("TripletLoss initialized.")

    num_epochs = 1
    warmup_steps = int(len(train_dataloader) * num_epochs * 0.1)

    print("\nStarting model fine-tuning...")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    with GpuUtilSampler() as sampler:
        model.fit(
            train_objectives=[(train_dataloader, train_loss)],
            epochs=num_epochs,
            warmup_steps=warmup_steps,
            output_path=None,
            show_progress_bar=True,
            use_amp=True,
        )
    fit_seconds = time.perf_counter() - t0
    record_step("model_fit_gpu_training", save_path, fit_seconds)
    peak_allocated_gb = torch.cuda.max_memory_allocated() / 1e9
    peak_reserved_gb = torch.cuda.max_memory_reserved() / 1e9
    print(f"GPU utilization during fit(): {sampler.summary()}")
    print(f"Peak GPU memory during fit(): allocated={peak_allocated_gb:.2f}GB, reserved={peak_reserved_gb:.2f}GB")

    t0 = time.perf_counter()
    model.save(save_path)
    record_step("model_checkpoint_save", save_path, time.perf_counter() - t0)
    print(f"\nFine-tuning complete. Model saved to '{save_path}'.")

    t0 = time.perf_counter()
    anchor_embedding = model.encode(triplet_examples[0].texts[0])
    positive_embedding = model.encode(triplet_examples[0].texts[1])
    negative_embedding = model.encode(triplet_examples[0].texts[2])
    dist_positive = util.pytorch_cos_sim(anchor_embedding, positive_embedding).item()
    dist_negative = util.pytorch_cos_sim(anchor_embedding, negative_embedding).item()
    record_step("verification_step", save_path, time.perf_counter() - t0)
    print("\n--- Model Verification ---")
    print(f"Cosine Similarity (Anchor, Positive): {dist_positive:.4f}")
    print(f"Cosine Similarity (Anchor, Negative): {dist_negative:.4f}")

    del train_loss, train_dataloader
    gc.collect()
    torch.cuda.empty_cache()
    return model

# =========================
# Inference (rule-scoped semantic search)
# =========================
def get_scores(test_dataframe, embedding_model, model_label, corpus_dataframe):
    t0 = time.perf_counter()
    corpus_dataframe = corpus_dataframe.copy()
    corpus_dataframe["prompt"] = corpus_dataframe["body"]
    corpus_dataframe["rule_violation"] = corpus_dataframe["rule_violation"].map({1: 1, 0: -1})
    record_step("corpus_prep_copy", model_label, time.perf_counter() - t0)

    encode_query_total = 0.0
    encode_document_total = 0.0
    semantic_search_total = 0.0
    score_apply_total = 0.0

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc="Generate scores for each rule"):
        test_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_part = corpus_part.reset_index(names="row_id")

        t0 = time.perf_counter()
        query_embeddings = embedding_model.encode(
            sentences=test_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        encode_query_total += time.perf_counter() - t0

        t0 = time.perf_counter()
        document_embeddings = embedding_model.encode(
            sentences=corpus_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        encode_document_total += time.perf_counter() - t0

        t0 = time.perf_counter()
        test_part["semantic"] = semantic_search(
            query_embeddings, document_embeddings, top_k=TOP_K, score_function=dot_score,
        )
        semantic_search_total += time.perf_counter() - t0

        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_part[["row_id", "rule_violation"]],
                how="left", left_on="corpus_id", right_on="row_id",
            )
            semantic["score"] = semantic["score"] * semantic["rule_violation"]
            return semantic["score"].sum()

        tqdm.pandas(desc=f"Add label for {rule=}")
        t0 = time.perf_counter()
        test_part["rule_violation"] = test_part["semantic"].progress_apply(get_score)
        score_apply_total += time.perf_counter() - t0

        result.append(test_part[["row_id", "rule_violation"]].copy())

    record_step("encode_query_total", model_label, encode_query_total)
    record_step("encode_document_total", model_label, encode_document_total)
    record_step("semantic_search_total", model_label, semantic_search_total)
    record_step("score_apply_total", model_label, score_apply_total)

    del embedding_model
    gc.collect()
    torch.cuda.empty_cache()

    return pd.concat(result, axis=0)

# =========================
# Main: fine-tune + score ONE embedding model (given via CLI args), write its own CSV
# =========================
def main():
    if len(sys.argv) < 4:
        print("Usage: python train_embedding_single.py <model_path> <save_path> <output_csv>")
        sys.exit(1)
    model_path = sys.argv[1]
    save_path = sys.argv[2]
    output_csv = sys.argv[3]

    script_start = time.perf_counter()
    print_environment_diagnostics()

    t0 = time.perf_counter()
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)
    record_step("load_prepare_test_dataframe", save_path, time.perf_counter() - t0)

    t0 = time.perf_counter()
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        corpus_dataframe["body"] = corpus_dataframe["body"].progress_apply(cleaner)
    record_step("load_clean_corpus", save_path, time.perf_counter() - t0)

    train_start = time.perf_counter()
    model = train_embedding_model(model_path, save_path, corpus_dataframe)
    train_time_sec = time.perf_counter() - train_start

    infer_start = time.perf_counter()
    submission = get_scores(test_dataframe, model, save_path, corpus_dataframe)
    infer_time_sec = time.perf_counter() - infer_start

    submission = test_dataframe[["row_id", "rule"]].merge(submission, on="row_id", how="left")

    grp = submission.groupby("rule")
    rank = grp["rule_violation"].rank(method="average", ascending=True)
    n = grp["rule_violation"].transform("size")
    submission["rule_violation"] = (rank - 1.0) / np.maximum(n - 1.0, 1.0)

    submission[["row_id", "rule", "rule_violation"]].sort_values("row_id").to_csv(output_csv, index=False)
    print(submission.head(10))
    print(f"Wrote {output_csv}")
    print(f"[{save_path}] train: {train_time_sec:.2f}s | inference: {infer_time_sec:.2f}s")

    detailed_df = pd.DataFrame(STEP_TIMINGS)
    print("\n=== Detailed step-by-step timing log (seconds) ===")
    print(detailed_df.to_string(index=False))

    total_script_seconds = time.perf_counter() - script_start
    accounted_for = detailed_df["seconds"].sum()
    print(f"\n=== TOTAL SCRIPT WALL-CLOCK TIME: {total_script_seconds:.2f}s ===")
    print(f"Sum of all measured steps: {accounted_for:.2f}s "
          f"(unaccounted/overhead: {total_script_seconds - accounted_for:.2f}s)")

if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_cross_encoder_single.py
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ["WANDB_DISABLED"] = "true"
print("PID:", os.getpid(), "CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
import torch
assert torch.cuda.is_available()
print("Visible device count:", torch.cuda.device_count())
torch.cuda.set_device(0)            # 0 == the only GPU visible to this process
print("Using:", torch.cuda.get_device_name(0))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

import sys
import random
import time
import numpy as np
import pandas as pd
from datetime import datetime
from cleantext import clean
from tqdm.auto import tqdm
from datasets import Dataset
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.cross_encoder.losses.CrossEntropyLoss import CrossEntropyLoss
from sentence_transformers.cross_encoder.trainer import CrossEncoderTrainer
from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from constants import DATA_PATH, CROSS_ENCODER_BATCH_SIZE, CROSS_ENCODER_EPOCHS, CROSS_ENCODER_LEARNING_RATE, CROSS_ENCODER_MAX_LENGTH

CLEAN_TEXT = True


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv")
    flatten = [train_dataset[["body", "rule", "subreddit", "rule_violation"]]]
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(subset=["body", "rule"], ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe = dataframe.copy()
    dataframe["prompt"] = dataframe["body"]
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)
    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map({1: 1, 0: 0})
    return dataframe


def prepare_cross_encoder_data(df, cross_rule_ratio=0.4, random_seed=42):
    """Sentence pairs + labels, with cross-rule negatives added so the model
    also learns that a violation of RULE A isn't automatically a violation
    of RULE B."""
    sentence_pairs, labels = [], []
    for _, row in df.iterrows():
        if pd.notna(row["body"]) and pd.notna(row["rule"]):
            sentence_pairs.append((f"Rule: {row['rule']}", f"Comment: {row['body']}"))
            labels.append(int(row["rule_violation"]))

    violations = df[df["rule_violation"] == 1].copy()
    if len(violations) > 0:
        for rule in violations["rule"].unique():
            this_rule_violations = violations[violations["rule"] == rule]
            other_rule_violations = violations[violations["rule"] != rule]
            if len(other_rule_violations) > 0:
                n_samples = min(int(len(this_rule_violations) * cross_rule_ratio), len(other_rule_violations))
                if n_samples > 0:
                    sampled = other_rule_violations.sample(n=n_samples, random_state=random_seed, replace=False)
                    for _, violation_row in sampled.iterrows():
                        sentence_pairs.append((f"Rule: {rule}", f"Comment: {violation_row['body']}"))
                        labels.append(0)
    return sentence_pairs, labels


def train_cross_encoder_model(model_path, save_path, corpus_dataframe, random_seed):
    random.seed(random_seed)
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    torch.cuda.manual_seed_all(random_seed)

    sentence_pairs, labels = prepare_cross_encoder_data(corpus_dataframe, cross_rule_ratio=0.4, random_seed=random_seed)
    print(f"Generated {len(sentence_pairs)} training pairs")

    train_dataset = Dataset.from_dict({
        "sentence1": [p[0] for p in sentence_pairs],
        "sentence2": [p[1] for p in sentence_pairs],
        "label": labels,
    })

    model = CrossEncoder(model_path, num_labels=2, max_length=CROSS_ENCODER_MAX_LENGTH, device="cuda")
    loss = CrossEntropyLoss(model)

    # T4 (Turing, compute capability 7.5) has fast fp16 Tensor Cores but NO
    # bf16 Tensor Core acceleration -- that only exists on Ampere+ (compute
    # capability 8.0+). torch.cuda.is_bf16_supported() can't reliably tell
    # the two apart (it caused a real regression in a related experiment),
    # so check compute capability directly instead.
    major, _minor = torch.cuda.get_device_capability()
    use_bf16 = major >= 8

    args = CrossEncoderTrainingArguments(
        output_dir=f"output/reddit-cross-encoder-{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}",
        num_train_epochs=CROSS_ENCODER_EPOCHS,
        per_device_train_batch_size=CROSS_ENCODER_BATCH_SIZE,
        per_device_eval_batch_size=CROSS_ENCODER_BATCH_SIZE,
        warmup_ratio=0.1,
        learning_rate=CROSS_ENCODER_LEARNING_RATE,
        weight_decay=0.01,
        report_to="none",
        save_strategy="no",
        logging_steps=1000000,
        seed=random_seed,
        fp16=not use_bf16,
        bf16=use_bf16,
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
    )

    trainer = CrossEncoderTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
    trainer.train()

    final_output_dir = f"/kaggle/working/{save_path}"
    model.save_pretrained(final_output_dir)
    print(f"Model saved to: {final_output_dir}")
    return model


def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def get_cross_encoder_scores(test_dataframe, save_path):
    cross_encoder = CrossEncoder(f"/kaggle/working/{save_path}", max_length=CROSS_ENCODER_MAX_LENGTH, device="cuda")
    cross_encoder.model.half()   # inference-only fp16: pure speed win, no training-precision tradeoff

    pairs = list(zip(
        "Rule: " + test_dataframe["rule"].astype(str),
        "Comment: " + test_dataframe["prompt"].astype(str),
    ))
    predictions = cross_encoder.predict(pairs, batch_size=128, convert_to_numpy=True)
    if predictions.ndim == 2 and predictions.shape[1] == 2:
        scores = sigmoid(predictions[:, 1])
    else:
        scores = sigmoid(predictions.ravel())

    test_dataframe = test_dataframe.copy()
    test_dataframe["rule_violation"] = scores
    return test_dataframe[["row_id", "rule", "rule_violation"]]


def main():
    if len(sys.argv) < 5:
        print("Usage: python train_cross_encoder_single.py <model_path> <seed> <save_path> <output_csv>")
        sys.exit(1)
    model_path = sys.argv[1]
    seed = int(sys.argv[2])
    save_path = sys.argv[3]
    output_csv = sys.argv[4]

    t_start = time.perf_counter()

    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)

    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)

    t0 = time.perf_counter()
    train_cross_encoder_model(model_path, save_path, corpus_dataframe, seed)
    t_train = time.perf_counter() - t0

    t0 = time.perf_counter()
    submission = get_cross_encoder_scores(test_dataframe, save_path)
    t_infer = time.perf_counter() - t0

    # Rule-wise ranked score in [0, 1] for THIS member -- computed before
    # combining, so the ensemble averages calibrated per-rule ranks rather
    # than raw probabilities that may be calibrated differently per model.
    grp = submission.groupby("rule")
    rank = grp["rule_violation"].rank(method="average", ascending=True)
    n = grp["rule_violation"].transform("size")
    submission["rule_violation"] = (rank - 1.0) / np.maximum(n - 1.0, 1.0)

    submission[["row_id", "rule_violation"]].to_csv(output_csv, index=False)

    t_total = time.perf_counter() - t_start
    print("\n" + "=" * 50)
    print(f"CROSS-ENCODER TIMING (model_path={model_path}, seed={seed})")
    print("=" * 50)
    print(f"pred output path : {output_csv}")
    print(f"train time       : {t_train:.2f}s")
    print(f"infer time       : {t_infer:.2f}s")
    print(f"total wall time  : {t_total:.2f}s")
    print("=" * 50)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scheduler.py
"""Job allocator for Master_Ensemble.

Builds one Job per (model, seed) -- or per embedding/cross-encoder member --
from a selection, then schedules them across 2 GPUs either:
  - automatically (greedy longest-job-first, fills idle GPU time with
    whatever fits), or
  - from a manually-specified GPU assignment + run order.

Both produce the same `blocks` structure: a list of
{gpu, start, end, label, phase, model_id, unit} dicts, which the preview
and executor both consume the same way regardless of which mode built them.
"""
from dataclasses import dataclass

BUDGET_MIN = 12 * 60  # Kaggle competition run-time limit


@dataclass
class Job:
    model_id: str
    unit: object          # seed (int, LLM) or member label (str, embedding/cross-encoder)
    label: str
    train_min: float
    infer_min: float       # 0 for embedding/cross-encoder (combined single-stage run)
    infer_gpus: int         # 1 or 2 (embedding/cross-encoder are always 1)
    combined: bool = False  # True for embedding/cross-encoder: one process does both stages


def build_jobs(selection, model_registry, embedding_seeds=0, cross_encoder_members=0):
    """selection: {model_id: num_seeds} for LLM models (1-3 seeds each).
    embedding_seeds: 0-3, how many of the 3 embedding members to include.
    cross_encoder_members: 0-3, how many of the 3 cross-encoder members to include.
    """
    from constants import EMBEDDING_MEMBERS, EMBEDDING_INFO, CROSS_ENCODER_MEMBERS, CROSS_ENCODER_INFO

    jobs = []
    for model_id, n_seeds in selection.items():
        cfg = model_registry[model_id]
        seeds = [1010, 2020, 3030][:n_seeds]
        for seed in seeds:
            jobs.append(Job(
                model_id=model_id,
                unit=seed,
                label=f"{cfg['name']} #{seed}",
                train_min=cfg["train_min"],
                infer_min=cfg["infer_min"],
                infer_gpus=2 if cfg["tp_mode"] == "tensor_parallel" else 1,
            ))
    if embedding_seeds > 0:
        for member in EMBEDDING_MEMBERS[:embedding_seeds]:
            jobs.append(Job(
                model_id="embedding",
                unit=member["label"],
                label=f"Embedding {member['label']}",
                train_min=EMBEDDING_INFO["run_min"],
                infer_min=0,
                infer_gpus=1,
                combined=True,
            ))
    if cross_encoder_members > 0:
        for member in CROSS_ENCODER_MEMBERS[:cross_encoder_members]:
            jobs.append(Job(
                model_id="cross_encoder",
                unit=member["label"],
                label=f"Cross-Encoder {member['label']}",
                train_min=CROSS_ENCODER_INFO["run_min"],
                infer_min=0,
                infer_gpus=1,
                combined=True,
            ))
    return jobs


def auto_schedule(jobs):
    """Greedy longest-job-first scheduler across 2 GPUs."""
    gpu_free = [0.0, 0.0]
    blocks = []
    pending = [{"job": j, "status": "train", "infer_ready_at": None} for j in jobs]

    guard = 0
    while any(p["status"] != "done" for p in pending) and guard < 5000:
        guard += 1
        candidates = sorted(set([0.0, gpu_free[0], gpu_free[1]] +
                                 [p["infer_ready_at"] for p in pending
                                  if p["status"] == "infer" and p["infer_ready_at"] is not None]))
        progressed = False
        for t in candidates:
            free = [g for g in (0, 1) if gpu_free[g] <= t]
            if not free:
                continue
            ready = [p for p in pending if p["status"] == "train"
                     or (p["status"] == "infer" and p["infer_ready_at"] is not None and p["infer_ready_at"] <= t)]
            ready.sort(key=lambda p: -(p["job"].train_min + p["job"].infer_min))

            for p in ready:
                if not free:
                    break
                j = p["job"]
                if p["status"] == "train":
                    gpu = free.pop(0)
                    end = t + j.train_min
                    blocks.append({"gpu": gpu, "start": t, "end": end, "label": j.label,
                                    "phase": "train", "model_id": j.model_id, "unit": j.unit})
                    gpu_free[gpu] = end
                    if j.combined:
                        p["status"] = "done"
                    else:
                        p["status"] = "infer"
                        p["infer_ready_at"] = end
                    progressed = True
                elif p["status"] == "infer":
                    if j.infer_gpus == 1:
                        gpu = free.pop(0)
                        end = t + j.infer_min
                        blocks.append({"gpu": gpu, "start": t, "end": end, "label": j.label,
                                        "phase": "infer", "model_id": j.model_id, "unit": j.unit})
                        gpu_free[gpu] = end
                        p["status"] = "done"
                        progressed = True
                    elif j.infer_gpus == 2 and len(free) == 2:
                        end = t + j.infer_min
                        blocks.append({"gpu": 0, "start": t, "end": end, "label": j.label,
                                        "phase": "infer", "model_id": j.model_id, "unit": j.unit})
                        blocks.append({"gpu": 1, "start": t, "end": end, "label": j.label,
                                        "phase": "infer", "model_id": j.model_id, "unit": j.unit})
                        gpu_free[0] = gpu_free[1] = end
                        p["status"] = "done"
                        progressed = True
                        free = []
            if progressed:
                break
        if not progressed:
            break
    return blocks


def default_manual_template(jobs):
    """A reasonable starting point to edit: alternate GPUs, in job order."""
    template = []
    for i, j in enumerate(jobs):
        train_gpu = i % 2
        infer_gpus = [0, 1] if j.infer_gpus == 2 else [train_gpu]
        template.append({"model_id": j.model_id, "unit": j.unit,
                          "train_gpu": train_gpu, "infer_gpus": infer_gpus, "order": i})
    return template


def manual_schedule(jobs, assignments):
    """assignments: list of dicts, each
       {"model_id":..., "unit":..., "train_gpu": 0|1, "infer_gpus":[0]|[1]|[0,1], "order": int}
    Jobs are placed in ascending `order`; each stage starts as soon as its
    assigned GPU(s) are free AND (for infer) its own train has finished.
    """
    amap = {(a["model_id"], a["unit"]): a for a in assignments}
    gpu_free = [0.0, 0.0]
    blocks = []
    ordered_jobs = sorted(jobs, key=lambda j: amap[(j.model_id, j.unit)]["order"])

    for j in ordered_jobs:
        a = amap[(j.model_id, j.unit)]
        train_gpu = a["train_gpu"]
        train_start = gpu_free[train_gpu]
        train_end = train_start + j.train_min
        blocks.append({"gpu": train_gpu, "start": train_start, "end": train_end,
                        "label": j.label, "phase": "train", "model_id": j.model_id, "unit": j.unit})
        gpu_free[train_gpu] = train_end

        if j.combined:
            continue

        infer_gpus = a["infer_gpus"]
        infer_start = max(train_end, max(gpu_free[g] for g in infer_gpus))
        infer_end = infer_start + j.infer_min
        for g in infer_gpus:
            blocks.append({"gpu": g, "start": infer_start, "end": infer_end,
                            "label": j.label, "phase": "infer", "model_id": j.model_id, "unit": j.unit})
            gpu_free[g] = infer_end
    return blocks


def makespan(blocks):
    return max((b["end"] for b in blocks), default=0.0)


def fmt_min(m):
    m = round(m)
    h, mm = divmod(m, 60)
    if h == 0:
        return f"{mm}m"
    if mm == 0:
        return f"{h}h"
    return f"{h}h {mm}m"


def print_preview(blocks, title="Schedule preview"):
    total = makespan(blocks)
    print("=" * 60)
    print(title)
    print("=" * 60)
    for gpu in (0, 1):
        print(f"\nGPU {gpu}:")
        rows = sorted([b for b in blocks if b["gpu"] == gpu], key=lambda b: b["start"])
        if not rows:
            print("  (idle)")
        for b in rows:
            print(f"  {fmt_min(b['start']):>7} -> {fmt_min(b['end']):<7}  {b['label']:<28} [{b['phase']}]")
    print("\n" + "-" * 60)
    if total <= BUDGET_MIN:
        print(f"Total time: {fmt_min(total)}  -- fits within 12h limit ({fmt_min(BUDGET_MIN - total)} to spare)")
    else:
        print(f"Total time: {fmt_min(total)}  -- EXCEEDS 12h limit by {fmt_min(total - BUDGET_MIN)}")
    print("=" * 60)


def plot_schedule(blocks, title="Schedule preview"):
    import matplotlib.pyplot as plt
    total = makespan(blocks)
    display_max = max(total, BUDGET_MIN) * 1.05

    palette = ["#e0984a", "#4fb8b0", "#8b95e0", "#dd7f9e", "#a3bd6e", "#9099ac", "#c9a86a", "#7aa8c9"]
    ids = sorted(set(b["model_id"] for b in blocks))
    color_map = {mid: palette[i % len(palette)] for i, mid in enumerate(ids)}

    fig, ax = plt.subplots(figsize=(12, 2.6))
    for b in blocks:
        color = color_map[b["model_id"]]
        alpha = 0.45 if b["phase"] == "train" else 0.95
        ax.barh(b["gpu"], b["end"] - b["start"], left=b["start"], height=0.62,
                color=color, alpha=alpha, edgecolor="#222", linewidth=0.4)
        width = b["end"] - b["start"]
        if width > display_max * 0.03:
            ax.text(b["start"] + width / 2, b["gpu"], f"{b['label']}\n{b['phase']}",
                     ha="center", va="center", fontsize=6.5, color="#111")

    ax.axvline(BUDGET_MIN, color="#c94632", linestyle="--", linewidth=1.2, label="12h limit")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["GPU 0", "GPU 1"])
    ax.set_xlim(0, display_max)
    ax.set_xlabel("minutes")
    over = "  (OVER BUDGET)" if total > BUDGET_MIN else ""
    ax.set_title(f"{title}  —  total {fmt_min(total)}{over}")
    ax.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.show()

## 1. Choose models, seeds, mode, and blend weights

`SELECTION` picks which LLM approaches to run and how many seeds (1-3) each. `EMBEDDING_SEEDS` and `CROSS_ENCODER_MEMBERS_COUNT` (0-3 each) do the same for the embedding and cross-encoder ensembles' members. `MODE` picks the allocator strategy below. `WEIGHTS` controls how much each approach counts in the final blend (step 4) -- independent of how many seeds/members it runs.

In [ ]:
from constants import MODEL_REGISTRY
from scheduler import (
    build_jobs, auto_schedule, manual_schedule, default_manual_template,
    print_preview, plot_schedule, makespan, fmt_min, BUDGET_MIN,
)

# ---- pick your models + seed counts (1-3 seeds per LLM model) ----
SELECTION = {
    "lama3-8b":   3,
    "qwen3-8b":   3,
    "qwen3-4b":   3,
    "qwen25-7b":  3,
    "llama32-3b": 3,
}
EMBEDDING_SEEDS = 3                # 0-3 members of the embedding ensemble to include
CROSS_ENCODER_MEMBERS_COUNT = 3    # 0-3 members of the cross-encoder ensemble to include

# ---- pick the allocator mode ----
MODE = "AUTOMATIC"   # "AUTOMATIC" or "MANUAL"

# ---- per-approach blend weights (relative -- don't need to sum to 1) ----
# Used in step 4: each approach's own averaged score is multiplied by its
# weight, then the total is divided by the sum of weights of approaches
# actually included here -- so excluding an approach (SELECTION/seed count
# of 0) renormalizes the rest automatically instead of silently diluting them.
WEIGHTS = {
    "lama3-8b":      1.0,
    "qwen3-8b":      1.0,
    "qwen3-4b":      1.0,
    "qwen25-7b":     1.0,
    "llama32-3b":    1.0,
    "embedding":     1.0,
    "cross_encoder": 1.0,
}

jobs = build_jobs(SELECTION, MODEL_REGISTRY, EMBEDDING_SEEDS, CROSS_ENCODER_MEMBERS_COUNT)
print(f"{len(jobs)} job(s) built from {len(SELECTION)} LLM model(s)"
      + (f" + {EMBEDDING_SEEDS} embedding member(s)" if EMBEDDING_SEEDS else "")
      + (f" + {CROSS_ENCODER_MEMBERS_COUNT} cross-encoder member(s)" if CROSS_ENCODER_MEMBERS_COUNT else "")
      + f"  |  MODE = {MODE}")

## 2a. AUTOMATIC -- optimal GPU allocation

Greedy longest-job-first scheduler: fills idle GPU time with whatever job fits next, respecting that a seed's inference can't start before its own training finishes, and that tensor-parallel inference needs both GPUs exclusively.

In [ ]:
if MODE == "AUTOMATIC":
    blocks = auto_schedule(jobs)
    print_preview(blocks, title="AUTOMATIC schedule preview")
    plot_schedule(blocks, title="AUTOMATIC schedule")
else:
    print("MODE is not AUTOMATIC -- skipped.")

## 2b. MANUAL -- assign jobs yourself

`MANUAL_ASSIGNMENTS` starts as a simple alternating-GPU template -- edit `train_gpu` (0 or 1), `infer_gpus` ([0], [1], or [0,1] for tensor-parallel jobs), and `order` (run priority) per job, then re-run this cell to see the resulting timeline.

In [ ]:
if MODE == "MANUAL":
    MANUAL_ASSIGNMENTS = default_manual_template(jobs)

    # ---- edit assignments here, e.g.: ----
    # MANUAL_ASSIGNMENTS[0]["train_gpu"] = 1
    # MANUAL_ASSIGNMENTS[0]["order"] = 5

    blocks = manual_schedule(jobs, MANUAL_ASSIGNMENTS)
    print_preview(blocks, title="MANUAL schedule preview")
    plot_schedule(blocks, title="MANUAL schedule")
else:
    print("MODE is not MANUAL -- skipped.")

## 3. Execute

Runs the `blocks` schedule from whichever mode you used above for real: launches each step as its own subprocess pinned to its assigned GPU(s), waiting on precedence (a seed's inference waits for that seed's own training) and GPU availability (a step never starts on a GPU another step is still using). Uses real completion times, not the estimates from the preview.

In [ ]:
import subprocess
import time
from constants import EMBEDDING_MEMBERS, CROSS_ENCODER_MEMBERS, ce_save_dir_for, ce_pred_path_for


def blocks_to_steps(blocks):
    """Collapse per-GPU blocks back into steps -- a 2-GPU tensor-parallel
    infer produces 2 blocks (one per gpu) that are really ONE subprocess."""
    steps = {}
    for b in blocks:
        key = (b["model_id"], b["unit"], b["phase"])
        if key not in steps:
            steps[key] = {"model_id": b["model_id"], "unit": b["unit"], "phase": b["phase"],
                          "gpus": set(), "start": b["start"]}
        steps[key]["gpus"].add(b["gpu"])
    return sorted(steps.values(), key=lambda s: s["start"])


def launch_step(step):
    gpus = sorted(step["gpus"])
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in gpus)
    log_path = f"log_{step['model_id']}_{step['unit']}_{step['phase']}.log"

    if step["model_id"] == "embedding":
        member = next(m for m in EMBEDDING_MEMBERS if m["label"] == step["unit"])
        args = ["python", "-u", "train_embedding_single.py", member["model_path"], member["save_dir"], member["output_csv"]]
    elif step["model_id"] == "cross_encoder":
        member = next(m for m in CROSS_ENCODER_MEMBERS if m["label"] == step["unit"])
        args = ["python", "-u", "train_cross_encoder_single.py", member["model_path"], str(member["seed"]),
                 ce_save_dir_for(member["label"]), ce_pred_path_for(member["label"])]
    else:
        env["MODEL_ID"] = step["model_id"]
        env["SEED"] = str(step["unit"])
        script = "train_llm.py" if step["phase"] == "train" else "infer_llm.py"
        args = ["python", "-u", script]

    f = open(log_path, "w")
    proc = subprocess.Popen(args, env=env, stdout=f, stderr=subprocess.STDOUT)
    print(f"[launch] {step['model_id']} #{step['unit']} {step['phase']} on GPU{gpus} -> {log_path} (pid={proc.pid})")
    return proc, f


def execute_schedule(blocks, poll_seconds=5):
    steps = blocks_to_steps(blocks)
    remaining = list(steps)
    completed = set()
    gpu_busy = {0: None, 1: None}
    running = {}   # key -> (step, proc, logfile, gpus)
    t_start = time.perf_counter()

    while remaining or running:
        progressed = True
        while progressed:
            progressed = False
            for step in list(remaining):
                key = (step["model_id"], step["unit"], step["phase"])
                gpus = sorted(step["gpus"])
                dep_ok = step["phase"] != "infer" or (step["model_id"], step["unit"], "train") in completed
                gpu_ok = all(gpu_busy[g] is None for g in gpus)
                if dep_ok and gpu_ok:
                    proc, f = launch_step(step)
                    for g in gpus:
                        gpu_busy[g] = key
                    running[key] = (step, proc, f, gpus)
                    remaining.remove(step)
                    progressed = True

        if not running:
            break
        time.sleep(poll_seconds)
        for key, (step, proc, f, gpus) in list(running.items()):
            ret = proc.poll()
            if ret is not None:
                f.close()
                status = "OK" if ret == 0 else f"FAILED (exit {ret})"
                print(f"[done]   {step['model_id']} #{step['unit']} {step['phase']}: {status}")
                completed.add(key)
                for g in gpus:
                    gpu_busy[g] = None
                del running[key]

    print(f"\nAll steps complete in {time.perf_counter() - t_start:.1f}s wall time.")


execute_schedule(blocks)

## 4. Combine into `submission.csv`

Each seed/member already produces a per-rule ranked score in [0, 1]. Averages seeds/members within an approach first, then combines across approaches using `WEIGHTS` from step 1 (equal weights by default, so a model given 1 seed still contributes the same as one given 3 -- seed count controls that model's own stability, not its weight in the final blend, unless you change `WEIGHTS`).

In [ ]:
import pandas as pd
from constants import MODEL_REGISTRY, EMBEDDING_MEMBERS, CROSS_ENCODER_MEMBERS, pred_path_for, ce_pred_path_for

weighted_scores = []  # list of (weight, pd.Series) -- one entry per included approach

for model_id, n_seeds in SELECTION.items():
    if n_seeds <= 0:
        print(f"{MODEL_REGISTRY[model_id]['name']}: skipped (0 seeds selected)")
        continue
    seeds = [1010, 2020, 3030][:n_seeds]
    per_seed = [pd.read_csv(pred_path_for(model_id, s)).set_index("row_id")["rule_violation"] for s in seeds]
    avg_score = sum(per_seed) / len(per_seed)
    weight = WEIGHTS.get(model_id, 1.0)
    weighted_scores.append((weight, avg_score))
    print(f"{MODEL_REGISTRY[model_id]['name']}: averaged {len(seeds)} seed(s), weight={weight}")

if EMBEDDING_SEEDS > 0:
    members = EMBEDDING_MEMBERS[:EMBEDDING_SEEDS]
    per_member = [pd.read_csv(m["output_csv"])[["row_id", "rule_violation"]].set_index("row_id")["rule_violation"] for m in members]
    avg_score = sum(per_member) / len(per_member)
    weight = WEIGHTS.get("embedding", 1.0)
    weighted_scores.append((weight, avg_score))
    print(f"Embedding: averaged {len(members)} member(s), weight={weight}")

if CROSS_ENCODER_MEMBERS_COUNT > 0:
    members = CROSS_ENCODER_MEMBERS[:CROSS_ENCODER_MEMBERS_COUNT]
    per_member = [pd.read_csv(ce_pred_path_for(m["label"])).set_index("row_id")["rule_violation"] for m in members]
    avg_score = sum(per_member) / len(per_member)
    weight = WEIGHTS.get("cross_encoder", 1.0)
    weighted_scores.append((weight, avg_score))
    print(f"Cross-Encoder: averaged {len(members)} member(s), weight={weight}")

total_weight = sum(w for w, _ in weighted_scores)
final = sum(w * s for w, s in weighted_scores) / total_weight
submission = final.rename("rule_violation").reset_index().sort_values("row_id").reset_index(drop=True)
submission.to_csv("submission.csv", index=False)
print(f"\nCombined {len(weighted_scores)} approach(es) (total weight={total_weight}) into submission.csv ({len(submission)} rows)")

In [ ]:
!head submission.csv